# Model Analysis — Aeroponics Vision Model

This notebook analyzes the YOLO model bundled with the ML service
(`vision-aeroponik-model-test.pt`). It covers:

1. Environment & dependency check
2. Loading the trained weights
3. Architecture & parameter analysis (size, FLOPs, layers)
4. Class / label analysis
5. Inference benchmark (latency & throughput)
6. Sample inference & detection statistics
7. Optional validation on a dataset (if available)
8. Summary

> All UI/output text is in English per project rules.
>
> **Note:** If you ever see
> `RuntimeError: Only a single TORCH_LIBRARY can be used to register the namespace triton`,
> it means `torch` was reloaded inside the same kernel (e.g. a cell was re-run after a
> failed import). **Restart the kernel (Kernel → Restart) and Run All once.**

## 1. Environment & Dependency Check

In [2]:
import sys
import os
from pathlib import Path

import numpy as np
import torch
from ultralytics import YOLO

print(f"Python        : {sys.version.splitlines()[0]}")
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device    : {torch.cuda.get_device_name(0)}")

MODEL_PATH = Path("models/vision-aeroponik-model-test.pt")
if not MODEL_PATH.exists():
    alt = Path(__file__).resolve().parent / "models" / "vision-aeroponik-model-test.pt"
    MODEL_PATH = alt if alt.exists() else MODEL_PATH

print(f"Model file    : {MODEL_PATH}")
print(f"Exists        : {MODEL_PATH.exists()}")

Python        : 3.12.3 (main, Jun 19 2026, 12:46:00) [GCC 13.3.0]
PyTorch       : 2.11.0+cu128
CUDA available : True
CUDA device    : NVIDIA GeForce GTX 1660 SUPER
Model file    : models/vision-aeroponik-model-test.pt
Exists        : True


## 2. Load the Model

In [3]:
model = YOLO(str(MODEL_PATH))

# The underlying PyTorch module. Ultralytics wraps the nn.Module in `model.model`.
torch_module = model.model
print("Loaded YOLO model successfully.")
print(f"Task          : {model.task}")
print(f"Model family  : see Architecture section (parsed from info)")
print(f"Type          : {type(torch_module).__name__}")

Loaded YOLO model successfully.
Task          : segment
Model family  : see Architecture section (parsed from info)
Type          : SegmentationModel


## 3. Architecture & Parameter Analysis

In [4]:
def count_parameters(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable


def model_file_size_mb(path):
    return Path(path).stat().st_size / (1024 ** 2)


import re
import io
import logging

total_params, trainable_params = count_parameters(torch_module)

# ultralytics emits the model summary through its LOGGER (logging); the handler
# keeps a reference to the original stream, so stdout/stderr redirection misses it.
# Attach a temporary capture handler to read GFLOPs & model family reliably.
try:
    from ultralytics.utils import LOGGER as _ULOGGER
except Exception:
    _ULOGGER = logging.getLogger("ultralytics")

_cap = io.StringIO()
_handler = logging.StreamHandler(_cap)
_prev_propagate = _ULOGGER.propagate
_ULOGGER.addHandler(_handler)
_ULOGGER.propagate = False
model.info(detailed=False)
_ULOGGER.removeHandler(_handler)
_ULOGGER.propagate = _prev_propagate
summary_text = _cap.getvalue()
print(summary_text)

m_gflops = re.search(r"([\d.]+)\s*GFLOPs", summary_text)
GFLOPS = float(m_gflops.group(1)) if m_gflops else None

m_family = re.search(r"(YOLO[\w-]+)", summary_text)
MODEL_FAMILY = m_family.group(1) if m_family else "n/a"

print("=== Parameter Summary ===")
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"GFLOPs               : {GFLOPS}")
print(f"Model family         : {MODEL_FAMILY}")
print(f"Model file size      : {model_file_size_mb(MODEL_PATH):.2f} MB")

YOLO11m-seg summary: 254 layers, 22,381,575 parameters, 0 gradients, 113.6 GFLOPs
YOLO11m-seg summary: 254 layers, 22,381,575 parameters, 0 gradients, 113.6 GFLOPs

=== Parameter Summary ===
Total parameters     : 22,381,575
Trainable parameters : 0
GFLOPs               : 113.6
Model family         : YOLO11m-seg
Model file size      : 43.13 MB


In [5]:
# Layer composition: count layers by type to understand the backbone/head mix.
from collections import Counter

layer_counts = Counter(type(m).__name__ for m in torch_module.modules())
print("=== Layer Composition ===")
for name, count in layer_counts.most_common():
    if count > 0:
        print(f"{name:<28} {count}")

=== Layer Composition ===
Conv2d                       125
BatchNorm2d                  115
Conv                         109
Sequential                   26
Bottleneck                   16
ModuleList                   11
C3k2                         8
C3k                          8
DWConv                       6
Identity                     5
Concat                       4
Upsample                     2
SegmentationModel            1
SiLU                         1
SPPF                         1
MaxPool2d                    1
C2PSA                        1
PSABlock                     1
Attention                    1
Segment                      1
DFL                          1
Proto                        1
ConvTranspose2d              1


## 4. Class / Label Analysis

In [6]:
names = model.names
if isinstance(names, dict):
    num_classes = len(names)
    class_list = [names[i] for i in range(num_classes)]
else:
    num_classes = len(names)
    class_list = list(names)

print(f"Number of classes : {num_classes}")
for idx, name in enumerate(class_list):
    print(f"  class_id={idx:<3} -> {name}")

# Export as data for later cells.
NUM_CLASSES = num_classes
CLASS_NAMES = class_list

Number of classes : 29
  class_id=0   -> Umbi 1.1
  class_id=1   -> Umbi 1.2
  class_id=2   -> Umbi 1.3
  class_id=3   -> Umbi 1.4
  class_id=4   -> Umbi 1.5
  class_id=5   -> Umbi 2.1
  class_id=6   -> Umbi 2.2
  class_id=7   -> Umbi 3.1
  class_id=8   -> Umbi 3.2
  class_id=9   -> Umbi 3.3
  class_id=10  -> Umbi 3.4
  class_id=11  -> Umbi 3.5
  class_id=12  -> Umbi 5.1
  class_id=13  -> Umbi 5.2
  class_id=14  -> Umbi 5.3
  class_id=15  -> Umbi 6.1
  class_id=16  -> Umbi 6.2
  class_id=17  -> Umbi 8.1
  class_id=18  -> Umbi 8.2
  class_id=19  -> Umbi 8.3
  class_id=20  -> Umbi 9.1
  class_id=21  -> Umbi 9.2
  class_id=22  -> Umbi 9.3
  class_id=23  -> Umbi 10.1
  class_id=24  -> Umbi 16.1
  class_id=25  -> Umbi 16.2
  class_id=26  -> Umbi 17.1
  class_id=27  -> Umbi 18.1
  class_id=28  -> Umbi 19.1


## 5. Inference Benchmark

Generate synthetic RGB images and measure average latency / throughput at the
default input size. This is a load/performance probe, not an accuracy test.

In [7]:
import time
from PIL import Image

IMGSZ = 640
N_WARMUP = 3
N_ITERS = 20

# Synthetic inputs: random noise frames (no real plants, just a load test).
dummy_image = Image.fromarray(
    np.random.randint(0, 255, (IMGSZ, IMGSZ, 3), dtype=np.uint8)
)

device = 0 if torch.cuda.is_available() else "cpu"
print(f"Benchmark device : {device}")
print(f"Input size      : {IMGSZ}x{IMGSZ}")

# Warmup (CUDA lazy init / caching).
for _ in range(N_WARMUP):
    model.predict(dummy_image, imgsz=IMGSZ, conf=0.25, verbose=False)

latencies = []
for _ in range(N_ITERS):
    t0 = time.perf_counter()
    model.predict(dummy_image, imgsz=IMGSZ, conf=0.25, verbose=False)
    latencies.append((time.perf_counter() - t0) * 1000.0)

avg_ms = sum(latencies) / len(latencies)
min_ms = min(latencies)
max_ms = max(latencies)
p95_ms = sorted(latencies)[int(0.95 * len(latencies)) - 1]
fps = 1000.0 / avg_ms

print("\n=== Inference Benchmark (synthetic input) ===")
print(f"Iterations       : {N_ITERS}")
print(f"Avg latency      : {avg_ms:.2f} ms")
print(f"Min latency      : {min_ms:.2f} ms")
print(f"Max latency      : {max_ms:.2f} ms")
print(f"P95 latency      : {p95_ms:.2f} ms")
print(f"Throughput       : {fps:.2f} FPS")

Benchmark device : 0
Input size      : 640x640

=== Inference Benchmark (synthetic input) ===
Iterations       : 20
Avg latency      : 40.14 ms
Min latency      : 34.80 ms
Max latency      : 65.86 ms
P95 latency      : 57.29 ms
Throughput       : 24.91 FPS


## 6. Sample Inference & Detection Statistics

Run inference on a synthetic frame and summarize what the model produces
(bounding boxes; masks are also available for segmentation models).

In [ ]:
import pandas as pd

result = model.predict(dummy_image, imgsz=IMGSZ, conf=0.25, verbose=False)[0]

boxes = result.boxes
n_det = len(boxes)

has_masks = getattr(result, "masks", None) is not None
print(f"Detections on synthetic frame : {n_det}")
print(f"Segmentation masks present    : {has_masks}")

rows = []
if n_det > 0:
    names_map = result.names
    for b in boxes:
        cls_id = int(b.cls[0])
        rows.append({
            "class_id": cls_id,
            "class_name": names_map[cls_id],
            "confidence": round(float(b.conf[0]), 4),
            "x1": int(b.xyxy[0][0]),
            "y1": int(b.xyxy[0][1]),
            "x2": int(b.xyxy[0][2]),
            "y2": int(b.xyxy[0][3]),
        })

det_df = pd.DataFrame(rows)
if not det_df.empty:
    display(det_df)
    print("\nPer-class detection counts:")
    print(det_df["class_name"].value_counts())
    print(f"\nConfidence  min={det_df['confidence'].min():.4f} "
          f"max={det_df['confidence'].max():.4f} "
          f"mean={det_df['confidence'].mean():.4f}")
else:
    print("No objects detected (expected for random-noise input).")

## 7. Optional: Validation on a Dataset

If a YOLO-format dataset config (e.g. `data.yaml`) is available, the cell below
runs `model.val()` to compute real mAP / precision / recall. It is skipped when
no dataset is present.

In [ ]:
DATA_YAML = Path("models/data.yaml")  # adjust to your dataset config if present

if DATA_YAML.exists():
    metrics = model.val(data=str(DATA_YAML), imgsz=IMGSZ, verbose=False)
    print("=== Validation Metrics ===")
    print(f"mAP50-95 : {metrics.box.map:.4f}")
    print(f"mAP50    : {metrics.box.map50:.4f}")
    print(f"Precision: {metrics.box.mp:.4f}")
    print(f"Recall   : {metrics.box.mr:.4f}")
else:
    print(f"No dataset config found at {DATA_YAML}; skipping validation.")
    print("Provide a YOLO data.yaml (with `val`/`test` split) to enable mAP analysis.")

## 8. Summary

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Model file",
        "Model family",
        "Task",
        "File size (MB)",
        "Total parameters",
        "GFLOPs",
        "Number of classes",
        "Input size",
        "Avg latency (ms, synthetic)",
        "Throughput (FPS, synthetic)",
    ],
    "Value": [
        str(MODEL_PATH.name),
        MODEL_FAMILY,
        model.task,
        f"{model_file_size_mb(MODEL_PATH):.2f}",
        f"{total_params:,}",
        str(GFLOPS),
        NUM_CLASSES,
        f"{IMGSZ}x{IMGSZ}",
        f"{avg_ms:.2f}",
        f"{fps:.2f}",
    ],
})
display(summary)